In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import os

years = [2025] 
tables = ["advanced", "per_game", "totals"]

os.makedirs("wnba_data/raw_data", exist_ok=True)

for year in years:
    for table in tables:
        url = f"https://www.basketball-reference.com/wnba/years/{year}_{table}.html"
        page = requests.get(url)
        soup = BeautifulSoup(page.text, "html.parser")
        html_table = soup.find("table", {"id": table})

        rows = []
        for tr in html_table.find("tbody").find_all("tr", class_="full_table"):
            row = {}
            for td in tr.find_all(["th", "td"]):
                stat = td.get("data-stat")
                if stat == "player":
                    # Extract the player name from the <a> tag
                    a_tag = td.find("a")
                    row[stat] = a_tag.get_text(strip=True) if a_tag else td.get_text(strip=True)
                else:
                    row[stat] = td.get_text(strip=True)
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(f"wnba_data/raw_data/{year}_{table}.csv", index=False)
        print(f"{year} {table} saved ({len(df)} rows)")

        time.sleep(4)

2025 advanced saved (182 rows)
2025 per_game saved (182 rows)
2025 totals saved (182 rows)


In [2]:
print(df.head())
print(df.shape)
print(df.columns.tolist())

                player team  pos   g   mp  gs   fg  fga fg_pct fg3  ... fta  \
0  Monique Akoa Makani  PHO    G  41  881  40  114  265   .430  49  ...  41   
1       Julie Allemand  LAS    G  34  963  27   70  159   .440  28  ...  19   
2        Lindsay Allen  CON    G  31  450   9   26   65   .400   3  ...  23   
3        Rebecca Allen  CHI  F-G  44  821  17   82  240   .342  42  ...  31   
4     Laeticia Amihere  GSV    F  29  385   0   52  114   .456   3  ...  63   

  ft_pct orb  trb  ast stl blk tov   pf  pts  
0   .927  25   91  111  32   3  55  101  315  
1   .789  13  127  170  45   1  53   55  183  
2   .826   9   32   61  10   6  31   37   74  
3   .613  16  113   57  23  23  39   62  225  
4   .778  35  125   27  17  13  32   36  156  

[5 rows x 26 columns]
(182, 26)
['player', 'team', 'pos', 'g', 'mp', 'gs', 'fg', 'fga', 'fg_pct', 'fg3', 'fg3a', 'fg3_pct', 'fg2', 'fg2a', 'fg2_pct', 'ft', 'fta', 'ft_pct', 'orb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pf', 'pts']


In [3]:
url = "https://www.spotrac.com/wnba/rankings/salary/"
page = requests.get(url)
soup = BeautifulSoup(page.text, "html.parser")
soup

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">

<html><head><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<title>ERROR: The request could not be satisfied</title>
</head><body>
<h1>403 ERROR</h1>
<h2>The request could not be satisfied.</h2>
<hr noshade="" size="1px"/>
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.
<br clear="all"/>
If you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.
<br clear="all"/>
<hr noshade="" size="1px"/>
<pre>
Generated by cloudfront (CloudFront)
Request ID: O50z6qHuVN9WUXk0fjuFYR3BL0YhzUisLaOh8wm1rcc5DSAlZPyr7g==
</pre>
<address>
</address>
</body></html>

In [4]:
url = "https://www.basketball-reference.com/wnba/years/2025.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Print all table ids
for table in soup.find_all("table"):
    print(table.get("id"))

wnba_standings
standings_e
standings_w
per_game-team
per_game-opponent
totals-team
totals-opponent
advanced-team


In [5]:
years = [2025]
team_tables = ["wnba_standings", "advanced-team"]
os.makedirs("wnba_data/raw_data", exist_ok=True)

for year in years:
    url = f"https://www.basketball-reference.com/wnba/years/{year}.html"
    page = requests.get(url)
    soup = BeautifulSoup(page.text, "html.parser")

    for table in team_tables:
        html_table = soup.find("table", {"id": table})

        rows = []
        for tr in html_table.find("tbody").find_all("tr"):
            row = {}
            for td in tr.find_all(["th", "td"]):
                stat = td.get("data-stat")
                row[stat] = td.get_text(strip=True)
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(f"wnba_data/raw_data/{year}_{table}.csv", index=False)
        print(f"{year} {table} saved ({len(df)} rows)")

    time.sleep(4)

2025 wnba_standings saved (13 rows)
2025 advanced-team saved (13 rows)


In [6]:
url = "https://herhoopstats.com/salary-cap-sheet/wnba/players/salary_2025/stats_2024/"
page = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
print(page.status_code)
soup = BeautifulSoup(page.text, "html.parser")
for table in soup.find_all("table"):
    print(table.get("id"), table.get("class"))

200
None ['table', 'table-hover', 'table-condensed', 'table-responsive', 'sortable', 'salary-stat']


In [8]:
# Print the first row's raw HTML to see the structure
table = soup.find("table", {"class": "salary-stat"})
first_row = table.find("tbody").find("tr")
print(first_row.prettify())

<tr>
 <td class="research_stat_cell table_cell_left salary_player_name" sorttable_customkey="Wilson, A'ja">
  <a class="d-none d-sm-block" href="/stats/wnba/player/aja-wilson-stats-11eaecc7-44fc-b036-b611-2362f5011b0b/">
   A'ja Wilson
  </a>
  <a class="d-block d-sm-none" href="/stats/wnba/player/aja-wilson-stats-11eaecc7-44fc-b036-b611-2362f5011b0b/">
   A Wilson
  </a>
  <td class="roster_stat_cell table_cell_right salary_cap_hit salary_protected_veteran" data-html="true" data-placement="top" data-toggle="tooltip" sorttable_customkey="0200000-RFA-PV" title="Protected Veteran">
   $200,000
   <td class="research_stat_cell table_cell_center" data-html="true" data-placement="top" data-toggle="tooltip" sorttable_customkey="RFA-0200000" title="Restricted Free Agent">
    RFA
    <td class="research_stat_cell table_cell_right" sorttable_customkey="38">
     38
    </td>
    <td class="research_stat_cell table_cell_right" sorttable_customkey="38">
     38
    </td>
    <td class="research_

In [7]:
# Print header row to see column names
header_row = table.find("thead").find("tr")
print(header_row.prettify())

<tr>
 <th class="research_stat_header table_cell_left" data-html="true" data-placement="top" data-toggle="tooltip" title="">
  Player
 </th>
 <th class="research_stat_header table_cell_right" data-html="true" data-placement="top" data-toggle="tooltip" title="">
  2025 Salary
 </th>
 <th class="research_stat_header table_cell_center" data-html="true" data-placement="top" data-toggle="tooltip" title="">
  2025 Signing
 </th>
 <th class="research_stat_header table_cell_right" data-html="true" data-placement="top" data-toggle="tooltip" title="Games Played">
  G
 </th>
 <th class="research_stat_header table_cell_right" data-html="true" data-placement="top" data-toggle="tooltip" title="Games Started">
  GS
 </th>
 <th class="research_stat_header table_cell_right" data-html="true" data-placement="top" data-toggle="tooltip" title="Minutes Per Game">
  MIN
 </th>
 <th class="research_stat_header table_cell_right" data-html="true" data-placement="top" data-toggle="tooltip" title="Points Scored P

In [8]:
# Salary data from "Her Hoop Stats WNBA Salary Cap Database" (Add citation)
# All other data is from "Basketball Reference WNBA Season Pages" (Add citation)
salary_years = [2025]

for salary_year in salary_years:
    url = f"https://herhoopstats.com/salary-cap-sheet/wnba/players/salary_{salary_year}/stats_{salary_year}/"
    page = requests.get(url)
    soup = BeautifulSoup(page.text, "html.parser")

    table = soup.find("table", {"class": "salary-stat"})
    if table is None:
        print(f"{salary_year} not found")
        continue

    # get column names from header
    headers = [th.get_text(strip=True) for th in table.find("thead").find("tr").find_all("th")]

    # parse rows
    rows = []
    for tr in table.find("tbody").find_all("tr"):
        cells = tr.find_all("td")
        rows = []
        for tr in table.find("tbody").find_all("tr"):
            cells = tr.find_all("td")
            row = {}
            for i, td in enumerate(cells):
                if i >= len(headers):
                    continue
                col = headers[i]
                if "salary_player_name" in td.get("class", []):
                    a_tag = td.find("a", {"class": "d-none d-sm-block"})
                    row[col] = a_tag.get_text(strip=True) if a_tag else td.get_text(strip=True)
                elif "salary_cap_hit" in td.get("class", []):
                    # extract just the salary from sorttable_customkey "0200000-RFA-PV"
                    key = td.get("sorttable_customkey", "")
                    salary = key.split("-")[0]  # gets "0200000"
                    row[col] = int(salary) if salary.isdigit() else td.get_text(strip=True)
                elif i == 2:
                    # signing status "RFA", "Core", "--"
                    key = td.get("sorttable_customkey", "")
                    row[col] = key.split("-")[0] if key else td.get_text(strip=True)
                else:
                    row[col] = td.get_text(strip=True)
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(f"wnba_data/raw_data/salary_{salary_year}.csv", index=False)
    print(f"salary_{salary_year} saved ({len(df)} rows)")

    time.sleep(3)

salary_2025 saved (227 rows)


In [9]:
df.head()

,Player,2025 Salary,2025 Signing,G,GS,MIN,PTS,FGM,FGA,FG%,...,FTA,FT%,ORB,DRB,TRB,AST,TOV,STL,BLK,PF
0,A'ja Wilson,200000,RFA,40,40,31.2,23.4,8.3,16.4,50.5%,...,7.3,85.5%,2.3,7.9,10.2,3.1,2.2,1.6,2.3,1.9
1,Napheesa Collier,214284,,33,33,32.3,22.9,8.3,15.7,53.1%,...,5.2,90.6%,1.9,5.4,7.3,3.2,2.1,1.6,1.5,2.4
2,Kelsey Mitchell,269244,Core,44,44,31.4,20.2,7.2,15.7,45.6%,...,4.3,78.4%,0.4,1.3,1.8,3.4,1.8,0.9,0.2,2.0
3,Kelsey Plum,202000,Core,43,43,35.1,19.5,6.1,14.4,42.2%,...,5.7,89.3%,0.2,2.8,3.1,5.7,3.0,1.2,0.1,2.7
4,Paige Bueckers,78831,Rookie,36,36,33.3,19.2,7.2,15.1,47.7%,...,4.2,88.8%,0.7,3.3,3.9,5.4,2.0,1.6,0.5,2.3
